In [12]:
import os
import sys
import importlib.util

# 1. Примусово додаємо корінь проєкту в sys.path, щоб модулі бачили один одного через 'src'
# Якщо ми в AI_proj/notebooks, піднімаємося на рівень вище в AI_proj
project_root = os.path.abspath(os.path.join('..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Додатково додаємо саму папку src для страховки
src_path = os.path.join(project_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Робочі шляхи налаштовано. Корінь проєкту: {project_root}")

# 2. Визначаємо прямі шляхи до файлів
metrics_file_path = os.path.join(src_path, 'metrics.py')
loader_file_path = os.path.join(src_path, 'data_loader.py')

# 3. Прямий імпорт модуля метрик (metrics.py)
spec_metrics = importlib.util.spec_from_file_location("metrics_mod", metrics_file_path)
metrics_mod = importlib.util.module_from_spec(spec_metrics)
spec_metrics.loader.exec_module(metrics_mod)
calculate_fitness_v1 = metrics_mod.calculate_fitness_v1

# 4. Прямий імпорт модуля завантаження даних (data_loader.py)
spec_loader = importlib.util.spec_from_file_location("loader_mod", loader_file_path)
loader_mod = importlib.util.module_from_spec(spec_loader)
spec_loader.loader.exec_module(loader_mod) # Тепер тут помилки не буде!
load_telemetry_csv = loader_mod.load_telemetry_csv

print(" Модулі успішно підключені!")

Робочі шляхи налаштовано. Корінь проєкту: c:\Users\Gidroid\Desktop\AI_proj
 Модулі успішно підключені!


In [13]:
# Назва файлу логу, який ми аналізуємо 
current_log_file = "test_run_01.csv"

try:
    # Завантажуємо реальну таблицю через наш data_loader
    df_flight = load_telemetry_csv(current_log_file)
    
    # Витягуємо реальні кути нахилу дрона з датчика
    # Ми беремо 'pitch_deg', але якщо захочеш оцінювати нахил вбік — можна змінити на 'roll_deg'
    real_angles = df_flight['pitch_deg'].tolist()
    
    print(f"Успішно зчитано лог '{current_log_file}'")
    print(f"Кількість зафіксованих точок часу в тесті: {len(real_angles)}")
    print(f"Перші значення кутів з датчика: {real_angles[:5]}")

except Exception as e:
    print(f"Помилка: Не вдалося прочитати дані з файлу. Перевір, чи лежить файл у data/raw/. Деталі: {e}")

Успішно зчитано лог 'test_run_01.csv'
Кількість зафіксованих точок часу в тесті: 1
Перші значення кутів з датчика: [3.31]


In [14]:
# Передаємо реальний масив кутів у нашу Fitness-функцію
flight_score = calculate_fitness_v1(real_angles, target_angle=0.0)

print("=" * 50)
print(f"АНАЛІЗ РЕАЛЬНОГО ПОЛЬОТУ (Лог: {current_log_file})")
print("=" * 50)
print(f"Цільовий кут балансу: 0.0°")
print(f"Середній нахил за час тесту: {sum(real_angles)/len(real_angles):.4f}°")
print(f"ПІДСУМКОВА ОЦІНКА СТАБІЛЬНОСТІ: {flight_score:.2f} балів із 100")
print("=" * 50)

if flight_score > 85:
    print("Чудовий результат! Дрон майже не хитався. Ці коефіцієнти PID можна зберегти як успішні.")
elif flight_score > 50:
    print("Помірні коливання. Дрон тримається, але ШІ має підібрати параметри точніше.")
else:
    print("Тряска занадто сильна! Політ нестабільний, такі налаштування бракуємо.")

АНАЛІЗ РЕАЛЬНОГО ПОЛЬОТУ (Лог: test_run_01.csv)
Цільовий кут балансу: 0.0°
Середній нахил за час тесту: 3.3100°
ПІДСУМКОВА ОЦІНКА СТАБІЛЬНОСТІ: 80.14 балів із 100
Помірні коливання. Дрон тримається, але ШІ має підібрати параметри точніше.


In [18]:
import os
import importlib.util

# 1. Підключаємо модулі Генетичного алгоритму та Бази даних прямо тут
base_dir = os.path.abspath(os.path.join('..'))
src_path = os.path.join(base_dir, 'src')

# Завантажуємо оптимізатор ШІ
ga_path = os.path.join(src_path, 'ga_optimizer.py')
spec_ga = importlib.util.spec_from_file_location("ga_module", ga_path)
ga_module = importlib.util.module_from_spec(spec_ga)
spec_ga.loader.exec_module(ga_module)
GeneticPIDOptimizer = ga_module.GeneticPIDOptimizer

# Завантажуємо функції бази даних
db_path = os.path.join(src_path, 'database.py')
spec_db = importlib.util.spec_from_file_location("db_module", db_path)
db_module = importlib.util.module_from_spec(spec_db)
spec_db.loader.exec_module(db_module)
init_db = db_module.init_db
save_experiment_result = db_module.save_experiment_result

# 2. Перевіряємо базу даних та генеруємо кандидатів, щоб вони точно були в пам'яті
init_db()
optimizer = GeneticPIDOptimizer(population_size=5)
initial_population = optimizer.init_population()

# 3. БЕРЕМО ПАРАМЕТРИ ПЕРШОГО РЕАЛЬНОГО КАНДИДАТА ВІД ШІ
current_candidate = initial_population[0]

kp_real = current_candidate['kp']
ki_real = current_candidate['ki']
kd_real = current_candidate['kd']

# 4. Підставляємо оцінку, яку ми щойно порахували в Ноутбуці 2!
real_calculated_score = 80.14  
flight_status = "success"      # Дрон тримався, не впав
log_file = "test_run_01.csv"   # Файл, з якого ми зчитували дані

# 5. Записуємо цей історичний перший тест в SQL-базу даних
save_experiment_result(
    kp=kp_real, 
    ki=ki_real, 
    kd=kd_real, 
    fitness_version="v1.0", 
    fitness_score=real_calculated_score, 
    status=flight_status, 
    csv_filename=log_file
)

print(f" ЗАПИСАНО В БАЗУ ДАНИХ experiments.db:")
print(f"   PID Налаштування ШІ -> Kp: {kp_real}, Ki: {ki_real}, Kd: {kd_real}")
print(f"   Оцінка за політ  -> {real_calculated_score} балів (Помірні коливання)")

Базу даних успішно ініціалізовано за шляхом: C:\Users\Gidroid\Desktop\AI_proj\data\processed\experiments.db
Результат експерименту успешно збережено в SQL!
 ЗАПИСАНО В БАЗУ ДАНИХ experiments.db:
   PID Налаштування ШІ -> Kp: 0.717, Ki: 0.227, Kd: 1.204
   Оцінка за політ  -> 80.14 балів (Помірні коливання)


In [ ]:
import os
import sys
import importlib.util

# 1. Визначаємо базові шляхи
base_dir = os.path.abspath(os.path.join('..'))
src_path = os.path.join(base_dir, 'src')

# Перевіряємо, чи є корінь проєкту в системних шляхах
if base_dir not in sys.path:
    sys.path.insert(0, base_dir)

# 2. Прямий імпорт УСІХ необхідних модулів, щоб нічого не губилося
# а) Оптимізатор
ga_path = os.path.join(src_path, 'ga_optimizer.py')
spec_ga = importlib.util.spec_from_file_location("ga_module", ga_path)
ga_module = importlib.util.module_from_spec(spec_ga)
spec_ga.loader.exec_module(ga_module)
GeneticPIDOptimizer = ga_module.GeneticPIDOptimizer

# б) База даних
db_path = os.path.join(src_path, 'database.py')
spec_db = importlib.util.spec_from_file_location("db_module", db_path)
db_module = importlib.util.module_from_spec(spec_db)
spec_db.loader.exec_module(db_module)
init_db = db_module.init_db
save_experiment_result = db_module.save_experiment_result

# в) Завантажувач даних
loader_path = os.path.join(src_path, 'data_loader.py')
spec_loader = importlib.util.spec_from_file_location("loader_mod", loader_path)
loader_mod = importlib.util.module_from_spec(spec_loader)
spec_loader.loader.exec_module(loader_mod)
load_telemetry_csv = loader_mod.load_telemetry_csv

# г) Метрики оцінки
metrics_path = os.path.join(src_path, 'metrics.py')
spec_metrics = importlib.util.spec_from_file_location("metrics_mod", metrics_path)
metrics_mod = importlib.util.module_from_spec(spec_metrics)
spec_metrics.loader.exec_module(metrics_mod)
calculate_fitness_v1 = metrics_mod.calculate_fitness_v1


# =========================================================
# 🔄 АВТОМАТИЧНИЙ ЗАПУСК: ВІД СТВОРЕННЯ ШІ ДО ЗАПИСУ В БД
# =========================================================

# Крок 1: Ініціалізуємо БД та Генетичний алгоритм
init_db()
optimizer = GeneticPIDOptimizer(population_size=5)
initial_population = optimizer.init_population() # ТЕПЕР ВОНО ТОЧНО Є В ПАМ'ЯТІ!

# Крок 2: Вказуємо файл польоту
new_log_file = "test_run_01.csv" 

try:
    # Крок 3: Автоматично завантажуємо лог і рахуємо оцінку
    df_new_flight = load_telemetry_csv(new_log_file)
    new_angles = df_new_flight['pitch_deg'].tolist()
    auto_score = calculate_fitness_v1(new_angles)

    # Крок 4: Беремо першого кандидата, якого щойно згенерував ШІ
    current_candidate = initial_population[0]

    # Крок 5: Записуємо весь результат в базу даних SQL
    save_experiment_result(
        kp=current_candidate['kp'], 
        ki=current_candidate['ki'], 
        kd=current_candidate['kd'], 
        fitness_version="v1.0", 
        fitness_score=auto_score, 
        status="success", 
        csv_filename=new_log_file
    )

    print(f"    Зчитано файл: {new_log_file}")
    print(f"    Розрахована оцінка стабільності: {auto_score:.2f} балів із 100")
    print(f"    Параметри ШІ (Kp: {current_candidate['kp']}, Ki: {current_candidate['ki']}, Kd: {current_candidate['kd']}) записано в SQL!")

except Exception as e:
    print(f" Помилка під час автоматичного виконання: {e}")

Базу даних успішно ініціалізовано за шляхом: C:\Users\Gidroid\Desktop\AI_proj\data\processed\experiments.db
Результат експерименту успешно збережено в SQL!
🤖 ПОВНА АВТОМАТИЗАЦІЯ УСПІШНА:
   📂 Зчитано файл: test_run_01.csv
   🏆 Розрахована оцінка стабільності: 80.14 балів із 100
   💾 Параметри ШІ (Kp: 0.725, Ki: 0.575, Kd: 1.266) записано в SQL!
